In [ ]:
import os
from virtual_accelerator.models.facet2 import get_facet_bmad_model

os.environ["FACET2_LATTICE"] = "/sdf/data/ad/ard-online/cropp/facet2-lattice"

model = get_facet_bmad_model(
    track_beam=True,
    start_element="L0AFEND",
    end_element="PR10571",
)

In [ ]:
variable_names = list(model.supported_variables.keys())
print(f"Supported variables: {len(variable_names)}")
print("Example variable names:")
print(variable_names[:10])

## Predictions at elements

In [ ]:
import matplotlib.pyplot as plt

out = model.get(["s_ele", "a.beta", "b.beta", "e_tot"])
plt.plot(out["s_ele"], out["e_tot"] / 1e6, label="b")
plt.ylabel("Total energy (MeV)")
plt.xlabel("s (m)")

plt.figure()
plt.plot(out["s_ele"], out["a.beta"], label="a.beta")
plt.plot(out["s_ele"], out["b.beta"], label="b.beta")
plt.xlabel("s (m)")
plt.ylabel("Beta function")
plt.legend()

## Predictions using comb

In [ ]:
import matplotlib.pyplot as plt

out = model.get(["s", "x.beta", "y.beta"])

plt.figure()
plt.plot(out["s"], out["x.beta"], label="x.beta")
plt.plot(out["s"], out["y.beta"], label="y.beta")
plt.xlabel("s (m)")
plt.ylabel("Beta function")
plt.legend()

## Staged FACET-II model

In [ ]:
from virtual_accelerator.models.facet2 import get_facet_staged_model

staged_model = get_facet_staged_model(
    surrogate_inputs="machine", n_particles=10000, end_element="PR10711"
)

In [ ]:
import matplotlib.pyplot as plt

out = staged_model.get(["s", "x.beta", "y.beta", "s_ele", "a.beta", "b.beta"])
plt.plot(out["s"], out["x.beta"], label="tracked x.beta")
plt.plot(out["s"], out["y.beta"], label="tracked y.beta")
plt.plot(out["s_ele"], out["a.beta"], "--", label="design a.beta")
plt.plot(out["s_ele"], out["b.beta"], "--", label="design b.beta")
plt.xlabel("s (m)")
plt.ylabel("Beta function")
plt.legend()

## FACET-II Impact


In [ ]:
from virtual_accelerator.models.facet2 import get_facet_impact_model
import os

os.environ["FACET2_LATTICE"] = "/sdf/data/ad/ard-online/cropp/facet2-lattice"
model = get_facet_impact_model(n_particles=1000, end_element="PR10241")

In [ ]:
model.supported_variables

In [ ]:
print("model default BACT:", model.get_value("SOLN:IN10:111:BACT"))
print(
    "IMPACT value:",
    model.impact_model.simulator.ele["SOL10111"]["solenoid_field_scale"],
)
print("Setting a new value")
model.set({"SOLN:IN10:111:BCTRL": 0.45})
print("Reading it back:", model.get_value("SOLN:IN10:111:BACT"))
print(
    "Closing the loop with IMPACT:",
    model.impact_model.simulator.ele["SOL10111"]["solenoid_field_scale"],
)

### Solenoid Scan 

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

image_data = model.get_value("PROF:IN10:241:Image:ArrayData")
q_nominal = model.get_value("SOLN:IN10:111:BCTRL")

# scan quad +/- 10%
q_scan = np.linspace(q_nominal * 0.9, q_nominal * 1.1, 5)

for q in q_scan:
    model.set({"SOLN:IN10:111:BCTRL": q})
    image_data = model.get_value("PROF:IN10:241:Image:ArrayData")

    plt.figure()
    plt.imshow(np.array(image_data[400:600, 600:800]))
    plt.title(f"SOLN:IN10:111:BCTRL = {q}")
    plt.colorbar()
    plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.plot(
    model.get_value("stat:mean_z"),
    model.get_value("stat:sigma_x"),
    label="sigma_x",
)
ax.plot(
    model.get_value("stat:mean_z"),
    model.get_value("stat:sigma_y"),
    label="sigma_y",
)
ax.plot(
    model.get_value("stat:mean_z"),
    model.get_value("stat:sigma_z"),
    label="sigma_z",
)
ax.set_xlabel("Mean Z")
ax.set_ylabel("Sigma")
ax.legend()

In [ ]:
model.impact_model.simulator.plot()

In [ ]:
model.distgen_model.simulator.run().plot("x", "px")

## Staged model with Impact

In [ ]:
import matplotlib.pyplot as plt

# from virtual_accelerator.models.facet2 import (
#     get_facet_impact_model,
#     get_facet_bmad_model,
# )
# from lume.staged_model import StagedModel
from virtual_accelerator.registry import get_model
# from virtual_accelerator.registry.models import MODELS

# impact_model = get_facet_impact_model(n_particles=1000, end_element="PR10241")

# facet_bmad_model = get_facet_bmad_model(
#     start_element="PR10241",
#     end_element="TD11",
#     track_beam=True,
# )

staged_model = get_model(
    ["impact_f2e_inj", "bmad_f2_elec"], end_ele="PR10711", n_particles=1000
)

In [ ]:
# plot the staged model
info = staged_model.get(
    [
        "s",
        "s_ele",
        "p0c",
        "stat:mean_z",
        "stat:mean_kinetic_energy",
        "stat:norm_emit_x",
        "x.norm_emit",
    ]
)

fig, ax = plt.subplots(2, 1, sharex=True)

ax[0].plot(info["s_ele"], info["p0c"])
ax[0].plot(info["stat:mean_z"], info["stat:mean_kinetic_energy"])
ax[0].set_ylabel("Kinetic Energy")
ax[1].plot(info["s"], info["x.norm_emit"], label="Bmad")
ax[1].plot(info["stat:mean_z"], info["stat:norm_emit_x"], label="Impact")
ax[1].set_xlabel("s")
ax[1].set_ylabel("Normalized Emittance x")
ax[1].legend()

In [ ]:
import numpy as np

s_tot = np.append(staged_model.get("stat:mean_z"), staged_model.get("s"))
rms_x = np.append(
    staged_model.get("stat:sigma_x"),
    (staged_model.get("x.emit") * staged_model.get("x.beta")) ** 0.5,
)
rms_y = np.append(
    staged_model.get("stat:sigma_y"),
    (staged_model.get("y.emit") * staged_model.get("y.beta")) ** 0.5,
)
plt.plot(s_tot, rms_x, label="RMS X")
plt.plot(s_tot, rms_y, label="RMS Y")
plt.legend()
plt.xlim(0, 30)
plt.xlabel("s")
plt.ylabel("RMS Beam Size")